# 06 · Post hoc image checks (plan 7f)

**Project:** Acquisition shortcuts in public cancer imaging (authors: see README.md)
**Notebook written:** 29 Oct 2026 · **Needs:** `results/01_inventory/` to `results/05_image_check/` · **Plan:** section
**7f**. **Post hoc:** these checks were decided after the notebook 05 results were seen. They change none of notebook
05's numbers, flags or wording; they only explain them.

**What this notebook does, in plain words**

In notebook 05, for two task–encoder pairs (LIDC with CT-FM, LUAD vs LUSC with Merlin), the image model predicted the
class even from the **background alone** (the body replaced by air). This notebook asks three follow-up questions:
1. **Inside one group:** does the background-only score still separate the classes inside one hospital (or vendor, or
   description language), and with whole groups held out? *(Notebook 05's saved embeddings; nothing downloaded.)*
2. **The body outline:** the background version still shows *where* the body was. A **silhouette** version shows only
   that region, as a flat grey shape on black: no table, no noise, no field-of-view edge. If the silhouette alone
   predicts the class, the background result may be the body outline. *(The scans are downloaded again.)*
3. **The DICOM header:** does the background score follow the acquisition that the header records?

**Rehearsal first.** With the default `RUN_MODE = "both"`, everything first runs on **shuffled labels** (a rehearsal:
every number should look like chance; it must also reproduce notebook 05's rehearsal exactly), checks itself, and only
then runs on the real labels.

**How to run:** in Colab choose *Runtime → Change runtime type → L4 GPU (or A100), High-RAM*, then *Runtime → Run all*.
Sign in with **the Google account that holds the project Drive folder** when Drive asks. Expect about 2.5–3.5 hours: about 1.5–2 hours to download,
prepare and embed the 525 scans again, then about 15 minutes per pass. **If Colab disconnects, run all again:** finished scans and
finished tasks are loaded, not redone. Outputs go to `results/06_image_posthoc/`.

In [ ]:
# 1. Setup: Drive, folders, GPU
import os, re, sys, json, time, copy, shutil, zipfile, hashlib, datetime, platform, warnings, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RUN_MODE = 'both'  # @param ["both", "rehearsal", "real"]

if IN_COLAB:
    PROJECT = Path('/content/drive/MyDrive/Acquisition-Shortcut-Audit')
    WORK = Path('/content/asa06_work')          # Colab's own disk: DICOM and NIfTI files, never Drive
else:  # local code tests only
    PROJECT = Path(os.environ['ASA_PROJECT_DIR'])
    WORK = Path(os.environ['ASA_WORK_DIR'])
    RUN_MODE = os.environ.get('ASA_TEST_RUN_MODE', 'rehearsal')
assert PROJECT.exists(), (f'Project folder not found: {PROJECT}. '
                          'Is Colab signed in to the Drive of the Google account that holds the project Drive folder?')
assert RUN_MODE in ('both', 'rehearsal', 'real'), RUN_MODE

IN01 = PROJECT / 'results' / '01_inventory'
IN02 = PROJECT / 'results' / '02_tasks_features'
IN03 = PROJECT / 'results' / '03_audit'
IN04 = PROJECT / 'results' / '04_posthoc'
IN05 = PROJECT / 'results' / '05_image_check'
CACHE05 = IN05 / 'cache'                         # notebook 05's label-free cache: headers, embeddings (read only)
OUT_ROOT = PROJECT / 'results' / '06_image_posthoc'
CACHE = OUT_ROOT / 'cache'                       # label-free: silhouette embeddings, preparation records, QC pictures
for d in (IN01, IN02, IN03, IN04):
    assert (d / 'run_manifest.json').exists(), f'{d} is missing: run the earlier notebooks first.'
for m in ('rehearsal', 'real'):
    assert (IN05 / m / 'run_manifest.json').exists(), f'{IN05 / m} is missing: run notebook 05 first.'
for d in (CACHE / 'embeddings', CACHE / 'prep', CACHE / 'qc', WORK / 'nifti', WORK / 'dicom'):
    d.mkdir(parents=True, exist_ok=True)

RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
free_gb = shutil.disk_usage(WORK).free / 1e9
print('Project folder:', PROJECT)
print('Outputs to    :', OUT_ROOT)
print('Work disk     :', WORK, f'({free_gb:.0f} GB free)')
print('Run mode      :', RUN_MODE)
print('Run started   :', RUN_STARTED, '(UTC)')
if free_gb < 80:
    print('WARNING: less than 80 GB free on the work disk. The scans need about 35 GB there. '
          'Use an L4 or A100 runtime (they have more disk).')

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Acquisition-Shortcut-Audit
Outputs to    : /content/drive/MyDrive/Acquisition-Shortcut-Audit/results/06_image_posthoc
Work disk     : /content/asa06_work (207 GB free)
Run mode      : both
Run started   : 2026-10-29T14:04:31+00:00 (UTC)


In [ ]:
# 2. Packages (exact versions; installed into Colab only) (notebook 05, unchanged)
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'monai==1.6.1', 'SimpleITK==2.5.6',
                    'pydicom==3.0.2', 'nibabel==5.4.2', 'merlin-vlm==0.0.7'], check=True)

import numpy as np
import pandas as pd
import requests
import scipy
from scipy import ndimage, sparse
from scipy.stats import rankdata, spearmanr
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
import joblib
from joblib import Parallel, delayed
import pydicom
import SimpleITK as sitk
import nibabel
import torch
import torchvision
import monai
from monai.transforms import (Compose, LoadImage, EnsureType, Orientation, Spacing, ScaleIntensityRange, SpatialCrop)
from monai.transforms.utils import generate_spatial_bounding_box
from monai.inferers import SlidingWindowSplitter
from monai.networks.nets.segresnet_ds import SegResEncoder
import huggingface_hub
from huggingface_hub import hf_hub_download
import safetensors.torch
import merlin
from merlin.models import i3res
from merlin.data.monai_transforms import ImageTransforms as MERLIN_TRANSFORMS
import importlib.metadata

warnings.filterwarnings('ignore', message='.*pixdim.*')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)
N_JOBS = -1
N_CPU = os.cpu_count()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def show(df, n=30):
    """Display a table in Colab, or print it in a plain Python run."""
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def write_json_atomic(path, obj):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, default=_jsonable))
    tmp.replace(path)

def _jsonable(o):
    return o.item() if hasattr(o, 'item') else str(o)

VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
            'scipy': scipy.__version__, 'scikit-learn': sklearn.__version__, 'joblib': joblib.__version__,
            'torch': torch.__version__, 'torchvision': torchvision.__version__, 'monai': monai.__version__,
            'SimpleITK': sitk.Version.VersionString(), 'pydicom': pydicom.__version__,
            'nibabel': nibabel.__version__, 'huggingface_hub': huggingface_hub.__version__,
            'merlin-vlm': importlib.metadata.version('merlin-vlm'), 'cpus': N_CPU,
            'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none'}
print(VERSIONS)
if IN_COLAB and DEVICE != 'cuda':
    print('WARNING: no GPU. Choose Runtime → Change runtime type → L4 GPU, then run all again.')

{'python': '3.13.16', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'joblib': '1.6.0', 'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'monai': '1.6.1', 'SimpleITK': '2.5.6', 'pydicom': '3.0.2', 'nibabel': '5.4.2', 'huggingface_hub': '1.33.0', 'merlin-vlm': '0.0.7', 'cpus': 12, 'gpu': 'NVIDIA L4'}


/usr/local/lib/python3.13/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


## Settings (plan sections 7d and 7f)
Every model, image and encoder setting is notebook 05's (checked against its run record below). The only new setting is
the silhouette's fill value.

In [ ]:
# 3. Settings (notebook 05's, unchanged) and the one new setting
SEED = 2026
N_SPLITS, N_REPEATS = 5, 5
N_BOOT = 2000
N_PERM = 999
MIN_COUNT = 3
FLAG_LOWER_CI, FLAG_Q = 0.60, 0.05
LR_PARAMS = dict(C=1.0, class_weight='balanced', max_iter=5000)
SCANNER_COLS = ['vendor', 'model', 'software_version', 'image_count', 'slice_thickness_mm', 'kernel', 'body_part',
                'protocol_text']
CATEGORICAL = {'vendor', 'model', 'software_version', 'kernel', 'body_part',
               'hdr_kernel', 'hdr_filter_type', 'hdr_patient_position'}
LOG_NUMERIC = {'image_count', 'body_area_cm2'}
TEXT = {'protocol_text', 'care_text'}
TASKS = {'renal subtype (KIRC vs KIRP)': 'hospital',
         'CPTAC-PDA | stage': 'description language',
         'lung histology (LUAD vs LUSC)': 'hospital',
         'LIDC-IDRI | primary vs metastatic': 'scanner vendor'}

TCIA = 'https://services.cancerimagingarchive.net/nbia-api/services/v1'
PREP_JOBS = max(1, min(8, N_CPU - 1))
EMB_WORKERS = max(1, min(6, N_CPU - 2)) if IN_COLAB else 0
BODY_HU, OPEN_MM, DILATE_MM, AIR_HU = -500, 3.0, 5.0, -1000
N_QC = 2
CTFM_REPO, CTFM_REVISION = 'project-lighter/ct_fm_feature_extractor', '527b0d1c262b433396c85ff2de84c427b26945a1'
CTFM_SPACING, CTFM_PATCH, CTFM_OVERLAP, CTFM_BATCH = (3.0, 1.0, 1.0), (24, 128, 128), 0.625, 32
MERLIN_REPO, MERLIN_REVISION = 'stanfordmimi/Merlin', '66dd07585de79e4d8d70d67806da88f3ccaa5a0c'
MERLIN_CKPT = 'i3_resnet_clinical_longformer_best_clip_04-02-2024_23-21-36_epoch_99.pt'
ENCODERS = ('ctfm', 'merlin')
HDR_NUMERIC = ['hdr_slice_thickness', 'hdr_z_spacing', 'hdr_pixel_spacing', 'hdr_recon_diameter', 'hdr_kvp',
               'hdr_tube_current', 'hdr_exposure', 'hdr_ctdivol', 'hdr_pitch']
HDR_CATEGORICAL = ['hdr_kernel', 'hdr_filter_type', 'hdr_patient_position']
HDR_COLS = HDR_NUMERIC + HDR_CATEGORICAL
GATE_NULL_RANGE = (0.35, 0.55)
GATE_MIN_COVERAGE = 0.95
# notebook 05's AUCs count as reproduced within this margin: with up to 2,048 features the logistic regression's solution
# differs in the last digits between machines (a local test on notebook 05's rehearsal: largest AUC difference 0.00004)
REPRO_TOL = 1e-3

# notebook 05's cache keys, recomputed with its exact formula (checked against its cached files below)
PREP_KEY05 = hashlib.md5(json.dumps({'BODY_HU': BODY_HU, 'OPEN_MM': OPEN_MM, 'DILATE_MM': DILATE_MM, 'AIR_HU': AIR_HU,
                                     'v': 1}, sort_keys=True).encode()).hexdigest()[:12]
EMB_KEY05 = hashlib.md5(json.dumps({'prep': PREP_KEY05, 'ctfm': [CTFM_REVISION, CTFM_SPACING, CTFM_PATCH, CTFM_OVERLAP],
                                    'merlin': [MERLIN_REVISION, MERLIN_CKPT], 'v': 1},
                                   sort_keys=True).encode()).hexdigest()[:12]

# New (plan 7f.2): the silhouette is the region the background version blanked (the dilated body mask), filled with
# 0 HU (water); everything else is air (-1000 HU).
SIL_HU = 0
VARIANTS = ('original', 'silhouette')       # embedded here: the original again (re-embedding check) and the silhouette
PREP_KEY = hashlib.md5(json.dumps({'nb05': PREP_KEY05, 'SIL_HU': SIL_HU, 'v': 1}, sort_keys=True).encode()).hexdigest()[:12]
EMB_KEY = hashlib.md5(json.dumps({'nb05': EMB_KEY05, 'SIL_HU': SIL_HU, 'v': 1}, sort_keys=True).encode()).hexdigest()[:12]

# Local code tests only (ignored in Colab)
TEST = {}
if not IN_COLAB:
    TEST = {'patients': [p for p in os.environ.get('ASA_TEST_PATIENTS', '').split(';') if p],
            'fake_images': os.environ.get('ASA_TEST_FAKE_IMAGES') == '1',
            'stop_after_images': os.environ.get('ASA_TEST_STOP_AFTER_IMAGES') == '1',
            'real_is_shuffled': True}   # local tests never see the real labels
    N_PERM = int(os.environ.get('ASA_TEST_N_PERM', N_PERM))
    N_BOOT = int(os.environ.get('ASA_TEST_N_BOOT', N_BOOT))
    CTFM_BATCH = int(os.environ.get('ASA_TEST_CTFM_BATCH', CTFM_BATCH))
    print('LOCAL TEST SETTINGS:', TEST, '| N_PERM', N_PERM, '| N_BOOT', N_BOOT)

In [ ]:
# 4. Load and verify the inputs from notebooks 01-05
def verify(folder):
    man = json.loads((folder / 'run_manifest.json').read_text())
    bad = [f['file'] for f in man['outputs'] if md5(folder / f['file']) != f['md5']]
    assert not bad, f'These files in {folder} changed since that run: {bad}'
    return man

man01, man02, man03, man04 = verify(IN01), verify(IN02), verify(IN03), verify(IN04)
man05 = {m: verify(IN05 / m) for m in ('rehearsal', 'real')}
for m, man in man05.items():
    assert man['in_colab'] and not man['local_test_settings'], f'notebook 05 {m} results are not the real Colab run'
    assert man['input_runs'] == {'01': man01['run_started_utc'], '02': man02['run_started_utc'],
                                 '03': man03['run_started_utc'], '04': man04['run_started_utc']}, \
        f'notebook 05 ({m}) was run on other inputs'
assert man05['real']['labels'] == 'real' and man05['rehearsal']['labels'].startswith('shuffled')
s05 = man05['real']['settings']
same = {'SEED': SEED, 'N_SPLITS': N_SPLITS, 'N_REPEATS': N_REPEATS, 'MIN_COUNT': MIN_COUNT,
        'FLAG_LOWER_CI': FLAG_LOWER_CI, 'FLAG_Q': FLAG_Q, 'LR_PARAMS': LR_PARAMS, 'BODY_HU': BODY_HU,
        'OPEN_MM': OPEN_MM, 'DILATE_MM': DILATE_MM, 'AIR_HU': AIR_HU, 'HDR_COLS': HDR_COLS, 'PREP_KEY': PREP_KEY05,
        'EMB_KEY': EMB_KEY05, 'CTFM': [CTFM_REPO, CTFM_REVISION, list(CTFM_SPACING), list(CTFM_PATCH), CTFM_OVERLAP],
        'MERLIN': [MERLIN_REPO, MERLIN_REVISION, MERLIN_CKPT]}
if not TEST:
    same.update({'N_BOOT': N_BOOT, 'N_PERM': N_PERM})
diff = [k for k, v in same.items() if json.loads(json.dumps(s05[k])) != json.loads(json.dumps(v))]
assert not diff, f'settings differ from notebook 05: {diff}'

# notebook 05's label-free cache must be exactly what its run recorded
cache05_files = sorted((q for q in CACHE05.rglob('*') if q.is_file() and q.suffix in ('.npz', '.json', '.csv')),
                       key=lambda q: q.relative_to(CACHE05).parts)
cache05_md5 = hashlib.md5(''.join(md5(q) for q in cache05_files).encode()).hexdigest()
assert (len(cache05_files), cache05_md5) == (man05['real']['cache_files']['count'],
                                             man05['real']['cache_files']['combined_md5']), \
    'notebook 05 cache differs from its run record'
v05 = man05['real']['versions']
same_versions = (v05['numpy'], v05['pandas'], v05['scipy'], v05['scikit-learn']) == \
                (np.__version__, pd.__version__, scipy.__version__, sklearn.__version__)
print('Inputs verified: notebooks 01-05 and the notebook 05 cache', f'({len(cache05_files)} files)')
print('Package versions', 'match notebook 05' if same_versions else
      f"DIFFER from notebook 05 (then numpy {v05['numpy']}, pandas {v05['pandas']}, scipy {v05['scipy']}, "
      f"scikit-learn {v05['scikit-learn']})")
probe05 = {m: pd.read_csv(IN05 / m / 'image_probe.csv') for m in ('rehearsal', 'real')}

labels = pd.read_csv(IN02 / 'labels_long.csv')
feat0 = pd.read_csv(IN02 / 'patient_features.csv', low_memory=False)
toks0 = pd.read_csv(IN02 / 'patient_tokens.csv.gz')
sites = pd.read_csv(IN02 / 'patient_sites.csv')
main = pd.read_csv(IN01 / 'main_series.csv', low_memory=False)
smeta = pd.read_csv(IN01 / 'series_metadata.csv.gz', usecols=['SeriesInstanceUID', 'FileSize']).drop_duplicates(
    'SeriesInstanceUID')
KEY = ['collection', 'PatientID', 'modality']
text = (toks0.assign(group=np.where(toks0.category == 'care pathway', 'care_text', 'protocol_text'))
             .groupby(KEY + ['group']).token.agg(lambda s: ' '.join(sorted(set(s)))).unstack('group').reset_index())
for c in ('protocol_text', 'care_text'):
    if c not in text:
        text[c] = ''
feat = feat0.merge(text, on=KEY, how='left')
for c in ('protocol_text', 'care_text'):
    feat[c] = feat[c].fillna('')
for c in ('vendor', 'model', 'software_version', 'kernel', 'body_part'):
    feat[c] = feat[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
site_of = sites.drop_duplicates('PatientID').set_index('PatientID').institution
POLISH_WORDS = set(man04['word_lists']['POLISH_WORDS'])
polish = toks0[toks0.token.isin(POLISH_WORDS)].drop_duplicates(KEY)[KEY].assign(polish=True)

lab = labels[labels.task_id.isin(TASKS)]
pts = lab.merge(main[KEY + ['SeriesInstanceUID', 'ImageCount']], on=KEY, how='left', validate='one_to_one')
pts = pts.merge(smeta, on='SeriesInstanceUID', how='left')
assert pts.SeriesInstanceUID.notna().all(), 'a task patient has no main series'
pts['key'] = pts.collection + '__' + pts.PatientID
PATIENTS = pts.drop_duplicates('key').sort_values('key').reset_index(drop=True)
if TEST.get('patients'):
    PATIENTS = PATIENTS[PATIENTS.key.isin(TEST['patients'])].reset_index(drop=True)
QC_KEYS = set(pts[pts.key.isin(PATIENTS.key)].sort_values('key').groupby('task_id').key.head(N_QC))
inv05 = pd.read_csv(CACHE05 / 'image_inventory.csv')
assert inv05.embedded.all() and set(inv05.key) == set(pts.key), 'notebook 05 did not embed every patient'
print(f"{len(PATIENTS)} patients, {PATIENTS.FileSize.sum() / 1e9:.1f} GB of DICOM to download again")

Inputs verified: notebooks 01-05 and the notebook 05 cache (1053 files)
Package versions match notebook 05
525 patients, 49.9 GB of DICOM to download again


## Part 1 · Download the scans again and make the silhouettes (no labels used)
Same series, same reading and same body mask as notebook 05. For each patient: the original volume and the silhouette
are saved on Colab's disk and embedded with both encoders. The original is embedded again only to check that this
session gives the same embeddings as notebook 05.

In [ ]:
# 5. Download, read and convert one series (notebook 05, unchanged)
def download_series(uid, dest):
    last = None
    for wait in (0, 10, 30, 90, 240):
        time.sleep(wait)
        try:
            tmp = dest.with_name(dest.name + '.part')
            with requests.get(f'{TCIA}/getImage', params={'SeriesInstanceUID': uid}, stream=True,
                              timeout=(30, 600)) as r:
                r.raise_for_status()
                with open(tmp, 'wb') as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            with zipfile.ZipFile(tmp) as z:
                badf = z.testzip()
                assert badf is None, f'corrupt file in zip: {badf}'
            tmp.replace(dest)
            return dest.stat().st_size
        except Exception as e:   # network errors, partial files: try again
            last = e
    raise RuntimeError(f'download failed 5 times: {last!r}')

def _num(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan

def _text(v):
    if v is None:
        return np.nan
    if isinstance(v, pydicom.multival.MultiValue):
        v = '/'.join(str(x) for x in v)
    v = str(v).strip().upper()
    return v if v else np.nan

def read_series(folder, uid):
    """The slices of one series, sorted along the slice normal, with geometry checks."""
    metas = []
    for p in sorted(folder.rglob('*')):
        if not p.is_file():
            continue
        try:
            ds = pydicom.dcmread(p, stop_before_pixels=True)
        except Exception:
            continue
        if str(getattr(ds, 'SeriesInstanceUID', '')) != uid:
            continue
        if not all(t in ds for t in ('ImagePositionPatient', 'ImageOrientationPatient', 'PixelSpacing', 'Rows')):
            continue
        metas.append((p, ds))
    assert metas, 'no image slices in the download'
    combo = [(tuple(np.round(np.asarray(ds.ImageOrientationPatient, float), 3)), int(ds.Rows), int(ds.Columns))
             for _, ds in metas]
    major = pd.Series(combo).value_counts().index[0]
    keep = [m for m, c in zip(metas, combo) if c == major]
    iop = np.asarray(keep[0][1].ImageOrientationPatient, float)
    row_cos, col_cos = iop[:3], iop[3:]
    normal = np.cross(row_cos, col_cos)
    pos = np.array([[float(v) for v in ds.ImagePositionPatient] for _, ds in keep])
    proj = pos @ normal
    inst = np.array([_num(getattr(ds, 'InstanceNumber', np.nan)) for _, ds in keep])
    order = np.lexsort((np.nan_to_num(inst, nan=1e9), proj))
    dup = np.r_[False, np.diff(proj[order]) < 1e-3]          # same position twice: keep the first
    order = order[~dup]
    keep, pos, proj = [keep[i] for i in order], pos[order], proj[order]
    assert len(keep) >= 10, f'only {len(keep)} usable slices'
    gaps = np.diff(proj)
    z = float(np.median(gaps))
    step = pos[-1] - pos[0]
    geo = {'row_cos': row_cos, 'col_cos': col_cos, 'normal': normal, 'origin': pos[0], 'z': z,
           'n_other_orientation_dropped': sum(c != major for c in combo),
           'n_duplicates_dropped': int(dup.sum()),
           'max_gap_deviation_mm': float(np.max(np.abs(gaps - z))),
           'nonuniform_spacing': bool(np.max(np.abs(gaps - z)) > max(0.05, 0.1 * z)),
           'gantry_tilt_deg': float(np.degrees(np.arccos(min(1.0, abs(step @ normal) / np.linalg.norm(step))))),
           'coverage_mm': float(proj[-1] - proj[0] + z)}
    return keep, geo

def header_fields(keep, geo):
    ds0 = keep[0][1]
    def med(name):
        vals = [_num(getattr(ds, name, None)) for _, ds in keep]
        vals = [v for v in vals if np.isfinite(v)]
        return float(np.median(vals)) if vals else np.nan
    ps = [float(v) for v in ds0.PixelSpacing]
    agent = getattr(ds0, 'ContrastBolusAgent', None)
    return {'hdr_manufacturer': _text(getattr(ds0, 'Manufacturer', None)),
            'hdr_model': _text(getattr(ds0, 'ManufacturerModelName', None)),
            'hdr_slice_thickness': med('SliceThickness'), 'hdr_z_spacing': geo['z'],
            'hdr_pixel_spacing': float(np.mean(ps)), 'hdr_rows': int(ds0.Rows), 'hdr_columns': int(ds0.Columns),
            'hdr_n_slices': len(keep), 'hdr_coverage_mm': geo['coverage_mm'],
            'hdr_recon_diameter': med('ReconstructionDiameter'), 'hdr_kvp': med('KVP'),
            'hdr_tube_current': med('XRayTubeCurrent'), 'hdr_exposure': med('Exposure'),
            'hdr_ctdivol': med('CTDIvol'), 'hdr_pitch': med('SpiralPitchFactor'),
            'hdr_kernel': _text(getattr(ds0, 'ConvolutionKernel', None)),
            'hdr_filter_type': _text(getattr(ds0, 'FilterType', None)),
            'hdr_patient_position': _text(getattr(ds0, 'PatientPosition', None)),
            'hdr_contrast_agent_given': ('yes' if agent is not None and str(agent).strip() else 'no'),
            'hdr_gantry_tilt_deg': geo['gantry_tilt_deg'], 'hdr_nonuniform_spacing': geo['nonuniform_spacing'],
            'hdr_max_gap_deviation_mm': geo['max_gap_deviation_mm'],
            'hdr_duplicates_dropped': geo['n_duplicates_dropped'],
            'hdr_other_orientation_dropped': geo['n_other_orientation_dropped']}

def to_image(keep, geo):
    """HU volume with the geometry from the DICOM positions (not from the reader's guess)."""
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames([str(p) for p, _ in keep])
    reader.SetOutputPixelType(sitk.sitkFloat32)          # GDCM applies the rescale slope and intercept (HU)
    img = reader.Execute()
    ps = [float(v) for v in keep[0][1].PixelSpacing]      # [between rows, between columns]
    img.SetSpacing((ps[1], ps[0], geo['z']))
    img.SetOrigin(tuple(float(v) for v in geo['origin']))
    img.SetDirection(tuple(np.stack([geo['row_cos'], geo['col_cos'], geo['normal']], axis=1).ravel()))
    return img

def like(arr, ref, dtype=np.int16):
    """A SimpleITK image from a (z, y, x) array, with the geometry of `ref`."""
    if dtype == np.int16:
        arr = np.clip(np.round(arr), -32768, 32767)
    out = sitk.GetImageFromArray(arr.astype(dtype))
    out.CopyInformation(ref)
    return out

In [ ]:
# 6. Body mask (notebook 05, unchanged), silhouette, QC picture
def body_mask(arr, sp):
    """arr: (z, y, x) HU; sp: (sz, sy, sx) mm. Returns (body, body dilated by DILATE_MM)."""
    m = arr > BODY_HU
    r = max(1, int(round(OPEN_MM / ((sp[1] + sp[2]) / 2))))
    yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    m = ndimage.binary_opening(m, structure=(yy ** 2 + xx ** 2 <= r * r)[None])
    lab_, n = ndimage.label(m)
    if n == 0:
        return np.zeros_like(m), np.zeros_like(m)
    counts = np.bincount(lab_.ravel())
    counts[0] = 0
    largest = lab_ == counts.argmax()
    del lab_, m
    core = ndimage.binary_fill_holes(largest, structure=np.ones((1, 3, 3), bool))
    del largest
    # dilation by DILATE_MM, computed inside the bounding box plus a margin (same result, much faster)
    pad = [int(np.ceil(DILATE_MM / s)) + 1 for s in sp]
    idx = np.where(core)
    lo = [max(0, int(i.min()) - p) for i, p in zip(idx, pad)]
    hi = [min(n_, int(i.max()) + p + 1) for i, n_, p in zip(idx, core.shape, pad)]
    box = tuple(slice(a, b) for a, b in zip(lo, hi))
    dil = np.zeros_like(core)
    dil[box] = ndimage.distance_transform_edt(~core[box], sampling=sp).astype(np.float32) <= DILATE_MM
    return core, dil

def silhouette(dil):
    """The region the background-only version blanked, as a flat shape: SIL_HU inside, air outside."""
    return np.where(dil, SIL_HU, AIR_HU).astype(np.int16)

def qc_picture(key, arr, core, bg, sil, path):
    from matplotlib.figure import Figure
    z = int(np.argmax(core.sum(axis=(1, 2))))
    fig = Figure(figsize=(12, 4.2))
    ax = fig.subplots(1, 3)
    kw = dict(cmap='gray', vmin=-1000, vmax=400)
    ax[0].imshow(arr[z], **kw); ax[0].contour(core[z], levels=[0.5], colors='r', linewidths=0.6)
    ax[0].set_title('original + body outline')
    ax[1].imshow(bg[z], **kw); ax[1].set_title('background only (notebook 05)')
    ax[2].imshow(sil[z], **kw); ax[2].set_title('silhouette (this notebook)')
    for a in ax:
        a.axis('off')
    fig.suptitle(key, fontsize=9)
    fig.tight_layout()
    fig.savefig(path, dpi=80)

def nifti_paths(key):
    d = WORK / 'nifti'
    return {'original': d / f'{key}__original.nii.gz', 'silhouette': d / f'{key}__silhouette.nii.gz'}

def prep_path(key):
    return CACHE / 'prep' / f'{key}.json'

def prep_ok(key):
    p = prep_path(key)
    return p.exists() and json.loads(p.read_text()).get('prep_key') == PREP_KEY

def header05(key):
    return json.loads((CACHE05 / 'headers' / f'{key}.json').read_text())

In [ ]:
# 7. Prepare one patient (runs in parallel workers)
def prepare_patient(key, uid, file_size):
    paths = nifti_paths(key)
    if prep_ok(key) and all(p.exists() for p in paths.values()):
        return {'key': key, 'status': 'ready'}
    t0 = time.time()
    tmp = WORK / 'dicom' / key
    zpath = WORK / 'dicom' / f'{key}.zip'
    try:
        shutil.rmtree(tmp, ignore_errors=True)
        download_series(uid, zpath)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(tmp)
        keep, geo = read_series(tmp, uid)
        img = to_image(keep, geo)
        arr = sitk.GetArrayFromImage(img)
        sp = img.GetSpacing()[::-1]
        core, dil = body_mask(arr, sp)
        sil = silhouette(dil)
        sitk.WriteImage(like(arr, img, np.int16), str(paths['original']), useCompression=True)
        sitk.WriteImage(like(sil, img, np.int16), str(paths['silhouette']), useCompression=True)
        area = core.sum(axis=(1, 2)) * sp[1] * sp[2] / 100.0
        h5 = header05(key)
        rec = {'key': key, 'SeriesInstanceUID': uid, 'prep_key': PREP_KEY, 'n_slices': len(keep),
               'body_area_cm2': float(area[area > 0].mean()) if (area > 0).any() else np.nan,
               'body_fraction_dilated': float(dil.mean()),
               'nb05_n_slices': h5.get('hdr_n_slices'), 'nb05_body_area_cm2': h5.get('body_area_cm2'),
               'nb05_body_fraction_dilated': h5.get('body_fraction_dilated'),
               'prep_seconds': round(time.time() - t0, 1)}
        rec['same_mask_as_nb05'] = bool(rec['n_slices'] == rec['nb05_n_slices'] and
                                        abs(rec['body_fraction_dilated'] - rec['nb05_body_fraction_dilated']) < 1e-9)
        if key in QC_KEYS:
            bg = arr.copy()
            bg[dil] = AIR_HU
            qc_picture(key, arr, core, bg, sil, CACHE / 'qc' / f'{key}.png')
        write_json_atomic(prep_path(key), rec)
        return {'key': key, 'status': 'ok', 'seconds': rec['prep_seconds']}
    except Exception as e:
        return {'key': key, 'status': 'failed', 'error': repr(e)[:400]}
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
        zpath.unlink(missing_ok=True)

### Encoders (notebook 05, unchanged)

In [ ]:
# 8. Load the two encoders (notebook 05, unchanged)
i3res.checkpoint = type('NoCheckpoint', (), {'checkpoint': staticmethod(lambda fn, *a, **k: fn(*a))})

def load_ctfm():
    cfg = json.loads(Path(hf_hub_download(CTFM_REPO, 'config.json', revision=CTFM_REVISION)).read_text())
    model = SegResEncoder(**cfg)
    state = safetensors.torch.load_file(hf_hub_download(CTFM_REPO, 'model.safetensors', revision=CTFM_REVISION))
    model.load_state_dict(state, strict=True)
    return model.eval()

MERLIN_DIR = Path(merlin.__file__).parent / 'models' / 'checkpoints'   # where the official class looks for it
if not IN_COLAB and os.environ.get('ASA_TEST_MERLIN_DIR'):              # local tests: Windows path-length limit
    MERLIN_DIR = Path(os.environ['ASA_TEST_MERLIN_DIR'])

def load_merlin():
    ckpt = hf_hub_download(MERLIN_REPO, MERLIN_CKPT, revision=MERLIN_REVISION, local_dir=MERLIN_DIR)
    state = torch.load(ckpt, map_location='cpu')
    prefix = 'encode_image.i3_resnet.'
    enc = i3res.I3ResNet(copy.deepcopy(torchvision.models.resnet152(weights=None)), class_nb=1692, conv_class=True,
                         ImageEmbedding=True)
    enc.load_state_dict({k[len(prefix):]: v for k, v in state.items() if k.startswith(prefix)}, strict=True)
    return enc.eval()

# CT-FM preprocessing (authors' script), split so that the background-only version can use the original's crop box
CTFM_LOAD = Compose([LoadImage(ensure_channel_first=True), EnsureType(), Orientation(axcodes='SPL'),
                     Spacing(pixdim=CTFM_SPACING, mode='bilinear')])
CTFM_SCALE = ScaleIntensityRange(a_min=-1024, a_max=2048, b_min=0, b_max=1, clip=True)

def ctfm_inputs(paths, crop_from=None):
    """Preprocessed CT-FM inputs for each variant. With crop_from, every variant uses that variant's foreground box."""
    imgs = {v: CTFM_LOAD(str(p)) for v, p in paths.items()}
    out, boxes = {}, {}
    for v, img in imgs.items():
        src = imgs[crop_from] if crop_from else img
        start, end = generate_spatial_bounding_box(src, select_fn=lambda x: x > 0)   # CropForeground's default
        out[v] = _plain(CTFM_SCALE(SpatialCrop(roi_start=start, roi_end=end)(img)))
        boxes[v] = [int(a) for a in start] + [int(b) for b in end]
    return out, boxes

def _plain(t):
    return (t.as_tensor() if hasattr(t, 'as_tensor') else torch.as_tensor(t)).float()

def merlin_inputs(paths):
    return {v: _plain(MERLIN_TRANSFORMS({'image': str(p)})['image']) for v, p in paths.items()}

class PrepDataset(torch.utils.data.Dataset):
    """CPU preprocessing in DataLoader workers; the GPU only runs the encoders."""
    def __init__(self, items):
        self.items = items              # (key, {variant: path}, crop_from)
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        key, paths, crop_from = self.items[i]
        try:
            ct, boxes = ctfm_inputs(paths, crop_from)
            return {'key': key, 'ctfm': ct, 'merlin': merlin_inputs(paths), 'boxes': boxes, 'error': None}
        except Exception as e:
            root = e
            while root.__cause__ is not None:      # MONAI wraps the real error; report the root cause
                root = root.__cause__
            return {'key': key, 'error': f'{e!r} | root cause: {root!r}'[:600]}

@torch.no_grad()
def ctfm_embed(model, vol):
    feats, batch = [], []
    def run(b):
        x = torch.cat(b).to(DEVICE)
        return torch.nn.functional.adaptive_avg_pool3d(model(x)[-1], 1).flatten(1).cpu()
    for patch, _ in SlidingWindowSplitter(CTFM_PATCH, CTFM_OVERLAP)(vol.unsqueeze(0)):
        batch.append(patch)
        if len(batch) == CTFM_BATCH:
            feats.append(run(batch)); batch = []
    if batch:
        feats.append(run(batch))
    f = torch.cat(feats)
    return f.mean(0).numpy().astype(np.float32), int(len(f))

@torch.no_grad()
def merlin_embed(model, vol):
    return model(vol.unsqueeze(0).to(DEVICE)).reshape(-1).cpu().numpy().astype(np.float32)

def embed_items(items, out_path_of, models, extra_of=None, label='scans'):
    """Embed every item with both encoders and save one .npz per item (resume point)."""
    loader = torch.utils.data.DataLoader(PrepDataset(items), batch_size=None, shuffle=False,
                                         num_workers=EMB_WORKERS if len(items) > 1 else 0)
    done, failed, t0 = 0, [], time.time()
    for rec in loader:
        if rec['error']:
            failed.append({'key': rec['key'], 'error': rec['error']})
            continue
        arrays, meta = {}, {'emb_key': EMB_KEY, 'boxes': rec['boxes'], **(extra_of(rec['key']) if extra_of else {})}
        for v, vol in rec['ctfm'].items():
            arrays[f'ctfm__{v}'], meta[f'ctfm__{v}__patches'] = ctfm_embed(models['ctfm'], vol)
        for v, vol in rec['merlin'].items():
            arrays[f'merlin__{v}'] = merlin_embed(models['merlin'], vol)
        path = out_path_of(rec['key'])
        tmp = path.with_name(path.stem + '.tmp.npz')
        np.savez(tmp, meta=json.dumps(meta, default=_jsonable), **arrays)
        tmp.replace(path)
        done += 1
        if done % 25 == 0 or done == len(items):
            print(f'  {label}: {done}/{len(items)} embedded ({(time.time() - t0) / 60:.1f} min)')
    return failed

def emb_path(key):
    return CACHE / 'embeddings' / f'{key}.npz'

def emb_ok(path, variants):
    if not path.exists():
        return False
    with np.load(path) as z:
        meta = json.loads(str(z['meta']))
        return meta.get('emb_key') == EMB_KEY and all(f'{e}__{v}' in z.files for e in ENCODERS for v in variants)

def emb05_path(key):
    return CACHE05 / 'embeddings' / f'{key}.npz'

/usr/local/lib/python3.13/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.array Orientation.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [9]:
# 9. Part 1: prepare and embed every scan (label-free; resumable), then the re-embedding check
def fake_part1():
    """Local code tests only: notebook 05's cached original and random silhouettes instead of images (label-free)."""
    rng = np.random.default_rng(7)
    for key in PATIENTS.key:
        if not emb_ok(emb_path(key), VARIANTS):
            with np.load(emb05_path(key)) as z:
                arrays = {f'{e}__original': z[f'{e}__original'] for e in ENCODERS}
            arrays.update({'ctfm__silhouette': rng.normal(size=512).astype(np.float32),
                           'merlin__silhouette': np.abs(rng.normal(size=2048)).astype(np.float32)})
            np.savez(emb_path(key), meta=json.dumps({'emb_key': EMB_KEY, 'fake': True}), **arrays)
        if not prep_ok(key):
            write_json_atomic(prep_path(key), {'key': key, 'prep_key': PREP_KEY, 'same_mask_as_nb05': True,
                                               'fake': True})

with np.load(emb05_path(PATIENTS.key[0])) as z:
    assert json.loads(str(z['meta']))['emb_key'] == EMB_KEY05, 'notebook 05 embeddings were made with other settings'

t_part1 = time.time()
prep_failed, emb_failed = [], []
if TEST.get('fake_images'):
    fake_part1()
    MODELS = None
else:
    todo = [r for r in PATIENTS.itertuples() if not emb_ok(emb_path(r.key), VARIANTS) or not prep_ok(r.key)]
    print(f'Scans to prepare: {len(todo)} of {len(PATIENTS)} (the others are finished in the cache)')
    if todo:
        res = Parallel(n_jobs=PREP_JOBS, return_as='generator_unordered')(
            delayed(prepare_patient)(r.key, r.SeriesInstanceUID, r.FileSize) for r in todo)
        for i, r in enumerate(res, 1):
            if r['status'] == 'failed':
                prep_failed.append(r)
                print(f"  FAILED {r['key']}: {r['error']}")
            if i % 25 == 0 or i == len(todo):
                print(f'  prepared {i}/{len(todo)} ({(time.time() - t_part1) / 60:.1f} min, '
                      f'{shutil.disk_usage(WORK).free / 1e9:.0f} GB free)')
    MODELS = {'ctfm': load_ctfm().to(DEVICE), 'merlin': load_merlin().to(DEVICE)}
    todo = [k for k in PATIENTS.key if not emb_ok(emb_path(k), VARIANTS)
            and all(p.exists() for p in nifti_paths(k).values())]
    print(f'Scans to embed: {len(todo)}')
    items = [(k, nifti_paths(k), 'original') for k in todo]     # the silhouette uses the original's CT-FM crop box
    emb_failed = embed_items(items, emb_path, MODELS, label='original + silhouette')
    for f in emb_failed:
        print(f"  EMBEDDING FAILED {f['key']}: {f['error']}")
print(f'Part 1 finished in {(time.time() - t_part1) / 60:.1f} min')

complete = {k for k in PATIENTS.key if emb_ok(emb_path(k), VARIANTS)}
rows = []
for k in sorted(complete):
    with np.load(emb_path(k)) as a, np.load(emb05_path(k)) as b:
        row = {'key': k}
        for e in ENCODERS:
            d = np.abs(a[f'{e}__original'] - b[f'{e}__original'])
            row[f'{e}_max_abs_diff'] = float(d.max())
            row[f'{e}_max_abs_value'] = float(np.abs(b[f'{e}__original']).max())
    p = json.loads(prep_path(k).read_text()) if prep_path(k).exists() else {}
    row['same_mask_as_nb05'] = p.get('same_mask_as_nb05')
    rows.append(row)
reemb = pd.DataFrame(rows, columns=['key', 'ctfm_max_abs_diff', 'ctfm_max_abs_value', 'merlin_max_abs_diff',
                                    'merlin_max_abs_value', 'same_mask_as_nb05'])
reemb.to_csv(CACHE / 'reembedding_check.csv', index=False)
pd.DataFrame(prep_failed + emb_failed, columns=['key', 'status', 'error']).to_csv(CACHE / 'failures_this_session.csv',
                                                                                  index=False)
cov = pts.assign(embedded=pts.key.isin(complete)).groupby('task_id').embedded.agg(['sum', 'size', 'mean'])
show(cov.rename(columns={'sum': 'embedded', 'size': 'patients', 'mean': 'coverage'}))
REEMB = {'patients': int(len(reemb)), 'same_mask_as_nb05': int(reemb.same_mask_as_nb05.eq(True).sum()),
         **{f'{e}_max_abs_diff': float(reemb[f'{e}_max_abs_diff'].max()) if len(reemb) else None for e in ENCODERS},
         **{f'{e}_median_max_abs_value': float(reemb[f'{e}_max_abs_value'].median()) if len(reemb) else None
            for e in ENCODERS}}
print('Re-embedding check (this session vs notebook 05):', REEMB)
if REEMB['same_mask_as_nb05'] < len(reemb):
    print('WARNING: for some scans the body mask differs from notebook 05 (see cache/reembedding_check.csv). Inspect the run log.')
if (cov['mean'] < GATE_MIN_COVERAGE).any():
    print(f'WARNING: fewer than {GATE_MIN_COVERAGE:.0%} of the patients of a task have silhouettes. '
          'Run all again (failed downloads are retried); if it persists, inspect the failed-download log.')
if TEST.get('stop_after_images'):
    raise SystemExit('Local test: stopping after the image part.')

Scans to prepare: 525 of 525 (the others are finished in the cache)
  prepared 25/525 (3.3 min, 205 GB free)
  prepared 50/525 (6.9 min, 202 GB free)
  prepared 75/525 (10.4 min, 201 GB free)
  prepared 100/525 (12.6 min, 200 GB free)
  prepared 125/525 (14.5 min, 199 GB free)
  prepared 150/525 (18.1 min, 196 GB free)
  prepared 175/525 (20.4 min, 195 GB free)
  prepared 200/525 (22.2 min, 195 GB free)
  prepared 225/525 (23.3 min, 195 GB free)
  prepared 250/525 (25.0 min, 194 GB free)
  prepared 275/525 (27.1 min, 192 GB free)
  prepared 300/525 (28.6 min, 192 GB free)
  prepared 325/525 (31.0 min, 190 GB free)
  prepared 350/525 (33.2 min, 189 GB free)
  prepared 375/525 (34.8 min, 189 GB free)
  prepared 400/525 (36.8 min, 188 GB free)
  prepared 425/525 (38.6 min, 187 GB free)
  prepared 450/525 (39.5 min, 187 GB free)
  prepared 475/525 (41.8 min, 184 GB free)
  prepared 500/525 (44.2 min, 183 GB free)
  prepared 525/525 (47.4 min, 183 GB free)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  311MB            

model.safetensors: downloading bytes:           |  0.00B            

i3_resnet_clinical_longformer_best_clip_(…): reconstructing file:   0%|          |  0.00B / 1.08GB            

i3_resnet_clinical_longformer_best_clip_(…): downloading bytes:           |  0.00B            

Scans to embed: 525
  original + silhouette: 25/525 embedded (5.9 min)
  original + silhouette: 50/525 embedded (10.8 min)
  original + silhouette: 75/525 embedded (15.8 min)
  original + silhouette: 100/525 embedded (20.0 min)
  original + silhouette: 125/525 embedded (24.6 min)
  original + silhouette: 150/525 embedded (28.6 min)
  original + silhouette: 175/525 embedded (34.1 min)
  original + silhouette: 200/525 embedded (41.4 min)
  original + silhouette: 225/525 embedded (47.6 min)
  original + silhouette: 250/525 embedded (57.1 min)
  original + silhouette: 275/525 embedded (65.3 min)
  original + silhouette: 300/525 embedded (73.1 min)
  original + silhouette: 325/525 embedded (81.1 min)
  original + silhouette: 350/525 embedded (87.0 min)
  original + silhouette: 375/525 embedded (94.3 min)
  original + silhouette: 400/525 embedded (100.4 min)
  original + silhouette: 425/525 embedded (106.6 min)
  original + silhouette: 450/525 embedded (109.5 min)
  original + silhouette: 47

,embedded,patients,coverage
task_id,,,
CPTAC-PDA | stage,68,68,1.0
LIDC-IDRI | primary vs metastatic,94,94,1.0
lung histology (LUAD vs LUSC),104,104,1.0
renal subtype (KIRC vs KIRP),259,259,1.0


Re-embedding check (this session vs notebook 05): {'patients': 525, 'same_mask_as_nb05': 525, 'ctfm_max_abs_diff': 0.0012521743774414062, 'merlin_max_abs_diff': 0.0008715391159057617, 'ctfm_median_max_abs_value': 207.5225067138672, 'merlin_median_max_abs_value': 1.50400972366333}


## Part 2 · The analysis (once on shuffled labels as a rehearsal, then on the real labels)

In [10]:
# 10. Model building blocks (identical to notebooks 03 and 04), plus the same models on embeddings (notebook 05, unchanged)
class TokenBinarizer(BaseEstimator, TransformerMixin):
    """One 0/1 column per word used by at least `min_count` training patients."""
    def __init__(self, min_count=MIN_COUNT):
        self.min_count = min_count
    def _docs(self, X):
        X = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(np.asarray(X).ravel())
        return X.fillna('').astype(str).tolist()
    def fit(self, X, y=None):
        counts = {}
        for d in self._docs(X):
            for t in set(d.split()):
                counts[t] = counts.get(t, 0) + 1
        self.vocab_ = {t: i for i, t in enumerate(sorted(t for t, c in counts.items() if c >= self.min_count))}
        return self
    def transform(self, X):
        rows, cols = [], []
        docs = self._docs(X)
        for i, d in enumerate(docs):
            for t in set(d.split()):
                j = self.vocab_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(docs), max(len(self.vocab_), 1)))

def column_transformer(cols):
    parts = []
    for c in cols:
        if c in CATEGORICAL:
            parts.append((c, Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='missing',
                                                              keep_empty_features=True)),
                                       ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=MIN_COUNT, sparse_output=True))]), [c]))
        elif c in TEXT:
            parts.append((c, TokenBinarizer(), [c]))
        else:
            steps = [('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one'))] if c in LOG_NUMERIC else []
            steps += [('fill', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
                      ('scale', StandardScaler())]
            parts.append((c, Pipeline(steps), [c]))
    return ColumnTransformer(parts, remainder='drop', sparse_threshold=0.3)

def auc_binary(y, s):
    """AUC by ranks (ties count one half); y in {0,1}."""
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def auc(y, P):
    k = P.shape[1]
    if k == 2:
        return auc_binary(y, P[:, 1])
    return float(np.nanmean([auc_binary((y == j).astype(int), P[:, j]) for j in range(k)]))

def repeated_splits(y, seed=SEED):
    return [list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed + r).split(np.zeros(len(y)), y))
            for r in range(N_REPEATS)]

def mean_auc(y, P):
    return float(np.mean([auc(y, P[r]) for r in range(P.shape[0])]))

def boot_ci(y, P):
    """Percentile 95% CI of the repeat-averaged AUC over patient bootstrap resamples."""
    rng = np.random.default_rng(SEED)
    k = P.shape[2]
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < k:
            continue
        vals.append(np.mean([auc(y[idx], P[r][idx]) for r in range(P.shape[0])]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_diff(y, PA, PB):
    """Repeat-averaged AUC(A) - AUC(B) on the same patients, with a paired bootstrap 95% CI."""
    rng = np.random.default_rng(SEED)
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(np.mean([auc(y[idx], PA[r][idx]) - auc(y[idx], PB[r][idx]) for r in range(PA.shape[0])]))
    return mean_auc(y, PA) - mean_auc(y, PB), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_mean(x):
    rng = np.random.default_rng(SEED)
    x = np.asarray(x, float)
    vals = [x[rng.choice(len(x), len(x), replace=True)].mean() for _ in range(N_BOOT)]
    return float(x.mean()), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

# --- metadata models (notebook 03) ---
def _fit_fold_df(df, y, cols, tr, te, k):
    m = Pipeline([('prep', column_transformer(cols)), ('model', LogisticRegression(**LR_PARAMS))]).fit(df.iloc[tr], y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(df.iloc[te])
    return te, proba

def cv_oof_df(df, y, cols, splits):
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_df)(df, y, cols, tr, te, k) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    for (r, _, _), (te, proba) in zip(jobs, res):
        P[r, te] = proba
    return P

# --- the same logistic regression on embeddings (standardized inside each training fold) ---
def _fit_fold_matrix(X, y, tr, te, k, extras):
    m = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(**LR_PARAMS))]).fit(X[tr], y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(X[te])
    out = {}
    for name, Xe in extras.items():           # the fitted fold model applied to other inputs of the same patients
        pe = np.full((len(te), k), np.nan)
        ok = np.where(~np.isnan(Xe[te]).any(axis=1))[0]
        if len(ok):
            tmp = np.zeros((len(ok), k))
            tmp[:, m.classes_] = m.predict_proba(Xe[te][ok])
            pe[ok] = tmp
        out[name] = pe
    return te, proba, out

def cv_oof_matrix(X, y, splits, extras=None):
    extras = extras or {}
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_matrix)(X, y, tr, te, k, extras) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    E = {n: np.full((len(splits), len(y), k), np.nan) for n in extras}
    for (r, _, _), (te, proba, out) in zip(jobs, res):
        P[r, te] = proba
        for n, pe in out.items():
            E[n][r, te] = pe
    return P, E

def _fit_fold_combined(df, E, y, cols, tr, te, k):
    ct = column_transformer(cols).fit(df.iloc[tr])
    sc = StandardScaler().fit(E[tr])
    Xtr = sparse.hstack([sparse.csr_matrix(ct.transform(df.iloc[tr])), sparse.csr_matrix(sc.transform(E[tr]))]).tocsr()
    Xte = sparse.hstack([sparse.csr_matrix(ct.transform(df.iloc[te])), sparse.csr_matrix(sc.transform(E[te]))]).tocsr()
    m = LogisticRegression(**LR_PARAMS).fit(Xtr, y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(Xte)
    return te, proba

def cv_oof_combined(df, E, y, cols, splits):
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_combined)(df, E, y, cols, tr, te, k) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    for (r, _, _), (te, proba) in zip(jobs, res):
        P[r, te] = proba
    return P

# --- exact permutation tests on one fixed label-independent split (notebook 03, 7b.6) ---
def encoded_folds_df(df, cols):
    out = []
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(len(df))):
        ct = column_transformer(cols).fit(df.iloc[tr])
        out.append((tr, te, ct.transform(df.iloc[tr]), ct.transform(df.iloc[te])))
    return out

def scaled_folds(X):
    out = []
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(X):
        sc = StandardScaler().fit(X[tr])
        out.append((tr, te, sc.transform(X[tr]), sc.transform(X[te])))
    return out

def _single_cv_proba(folds, y):
    k = int(y.max()) + 1
    P = np.zeros((len(y), k))
    for tr, te, Xtr, Xte in folds:
        m = LogisticRegression(**LR_PARAMS).fit(Xtr, y[tr])
        P[np.ix_(te, m.classes_)] = m.predict_proba(Xte)
    return P

def _perm_chunk(folds, ys):
    return [auc(yp, _single_cv_proba(folds, yp)) for yp in ys]

def permutation_test(folds, y):
    observed = auc(y, _single_cv_proba(folds, y))
    rng = np.random.default_rng(SEED)
    perms = [rng.permutation(y) for _ in range(N_PERM)]
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_perm_chunk)(folds, c) for c in chunks))
    return observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), null

# --- within-group AUC and its exact test (notebook 04, part D) ---
def within_site_auc(y, s, site):
    num, den, n_sites = 0.0, 0, 0
    for st in np.unique(site):
        m = site == st
        yy, ss = y[m], s[m]
        n1 = int(yy.sum()); n0 = len(yy) - n1
        if n1 and n0:
            r = rankdata(ss)
            num += r[yy == 1].sum() - n1 * (n1 + 1) / 2
            den += n1 * n0
            n_sites += 1
    return (num / den if den else np.nan), den, n_sites

def within_site_estimate(y, P, site):
    vals = [within_site_auc(y, P[r][:, 1], site)[0] for r in range(P.shape[0])]
    _, pairs, n_sites = within_site_auc(y, P[0][:, 1], site)
    if pairs == 0:
        return {'auc': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n_pairs': 0, 'n_sites_with_pairs': 0}
    rng = np.random.default_rng(SEED)
    boots = []
    groups = [np.where(site == st)[0] for st in np.unique(site)]
    while len(boots) < N_BOOT:
        idx = np.concatenate([rng.choice(g, len(g), replace=True) for g in groups])
        v = np.mean([within_site_auc(y[idx], P[r][idx, 1], site[idx])[0] for r in range(P.shape[0])])
        if not np.isnan(v):
            boots.append(v)
    return {'auc': float(np.mean(vals)), 'ci_low': float(np.percentile(boots, 2.5)),
            'ci_high': float(np.percentile(boots, 97.5)), 'n_pairs': int(pairs), 'n_sites_with_pairs': n_sites}

def _strata_chunk(folds, ys, strata, keep):
    return [within_site_auc(yp[keep], _single_cv_proba(folds, yp)[keep, 1], strata[keep])[0] for yp in ys]

def within_group_permutation_test(folds, y, strata, keep):
    observed = within_site_auc(y[keep], _single_cv_proba(folds, y)[keep, 1], strata[keep])[0]
    rng = np.random.default_rng(SEED)
    groups = [np.where(strata == s)[0] for s in np.unique(strata)]
    perms = []
    for _ in range(N_PERM):
        yp = y.copy()
        for g in groups:
            yp[g] = y[rng.permutation(g)]
        perms.append(yp)
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_strata_chunk)(folds, c, strata, keep) for c in chunks))
    return observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), null

def bh(p):
    """Benjamini-Hochberg q-values; missing p-values stay missing and are not counted."""
    p = np.asarray(p, float)
    q = np.full_like(p, np.nan)
    ok = np.where(~np.isnan(p))[0]
    order = ok[np.argsort(p[ok])]
    prev = 1.0
    for rank, j in reversed(list(enumerate(order, 1))):
        prev = min(prev, p[j] * len(ok) / rank)
        q[j] = prev
    return q

In [11]:
# 11. Notebook 05's task frame and embedding loader (unchanged), and group-held-out CV
from scipy.stats import kruskal

def task_frame(tid, shuffle_seed=None):
    """Task patients with metadata, groups, headers and body size. shuffle_seed: shuffled labels (rehearsal)."""
    df = labels[labels.task_id == tid].merge(feat, on=KEY, how='left', validate='one_to_one')
    df['key'] = df.collection + '__' + df.PatientID
    classes = sorted(df.label.unique())
    y = df.label.map({c: i for i, c in enumerate(classes)}).to_numpy()
    if shuffle_seed is not None:
        y = np.random.default_rng(shuffle_seed).permutation(y)
    df['site'] = df.PatientID.map(site_of)
    df = df.merge(polish, on=KEY, how='left')
    df['language'] = np.where(df.polish.eq(True), 'Polish', 'other')
    return df.reset_index(drop=True), y, classes

def load_embeddings(keys, variants, path_of):
    out = {(e, v): [] for e in ENCODERS for v in variants}
    for k in keys:
        p = path_of(k)
        ok = p is not None and p.exists()
        z = np.load(p) if ok else None
        for e in ENCODERS:
            for v in variants:
                name = f'{e}__{v}'
                out[(e, v)].append(z[name] if ok and name in z.files else None)
        if z is not None:
            z.close()
    res = {}
    for (e, v), lst in out.items():
        d = next((len(a) for a in lst if a is not None), 1)
        res[(e, v)] = np.vstack([a if a is not None else np.full(d, np.nan, np.float32) for a in lst]).astype(float)
    return res

def flag(lo, q):
    return bool(lo >= FLAG_LOWER_CI and q < FLAG_Q)

def heldout(X, y, strata, keep):
    """Whole groups held out (notebook 05's hospital-held-out CV; with two groups: train on one, test on the other)."""
    Xk, yk, sk = X[keep], y[keep], strata[keep]
    n_groups = len(np.unique(sk))
    out = {'n_groups': n_groups}
    if n_groups < 2:
        return {**out, 'note': 'fewer than two groups'}
    gkf = list(GroupKFold(min(N_SPLITS, n_groups)).split(np.zeros(len(yk)), yk, sk))
    if not all(len(np.unique(yk[tr])) == 2 for tr, _ in gkf):
        return {**out, 'note': 'a training fold had one class only'}
    Pg, _ = cv_oof_matrix(Xk, yk, [gkf])
    lo, hi = boot_ci(yk, Pg)
    return {**out, 'auc': mean_auc(yk, Pg), 'ci_low': lo, 'ci_high': hi}

def within_class_spearman(y, a, b, classes):
    out = {}
    for j, c in enumerate(classes):
        m = np.where(y == j)[0]
        rng = np.random.default_rng(SEED)
        boots = [spearmanr(a[bb], b[bb]).statistic for bb in (rng.choice(m, len(m), replace=True) for _ in range(N_BOOT))]
        out.update({f'class{j}': c, f'spearman_class{j}': float(spearmanr(a[m], b[m]).statistic),
                    f'spearman_class{j}_ci_low': float(np.nanpercentile(boots, 2.5)),
                    f'spearman_class{j}_ci_high': float(np.nanpercentile(boots, 97.5))})
    return out

In [12]:
# 12. Every analysis for one task (both encoders)
def analyse_task(tid, mode, shuffle_seed):
    df, y, classes = task_frame(tid, shuffle_seed)
    hdr_t = pd.DataFrame([header05(k) for k in df.key])
    wanted = ['body_area_cm2', 'hdr_coverage_mm', 'hdr_manufacturer'] + HDR_COLS
    df = df.merge(hdr_t[['key'] + [c for c in wanted if c in hdr_t]], on='key', how='left')
    for c in wanted:
        if c not in df:
            df[c] = np.nan
    for c in HDR_CATEGORICAL + ['hdr_manufacturer']:
        df[c] = df[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
    has = df.key.isin(complete).to_numpy()
    n_all = len(df)
    df, y = df[has].reset_index(drop=True), y[has]
    splits = repeated_splits(y)
    emb = {**load_embeddings(df.key, ('original', 'background'), emb05_path),
           **load_embeddings(df.key, ('silhouette',), emb_path)}
    grouping = TASKS[tid]
    strata = {'hospital': df.site.astype(object).where(df.site.notna(), '').astype(str).to_numpy(),
              'description language': df.language.to_numpy(),
              'scanner vendor': df.vendor.astype(object).where(df.vendor.notna(), '').astype(str).to_numpy()}[grouping]
    keep = strata != ''
    base = {'task_id': tid, 'n_patients': len(y), 'n_excluded_no_silhouette': int(n_all - len(y)),
            'classes': ' | '.join(f'{c}: {int((y == j).sum())}' for j, c in enumerate(classes))}
    P_hdr = cv_oof_df(df, y, HDR_COLS, splits)
    rec = {'silhouette': [], 'controlled': [], 'header': [], 'fields': [], 'oof': []}
    for enc in ENCODERS:
        X, Xb, Xs = emb[(enc, 'original')], emb[(enc, 'background')], emb[(enc, 'silhouette')]
        P, _ = cv_oof_matrix(X, y, splits)
        Pb, _ = cv_oof_matrix(Xb, y, splits)
        Ps, _ = cv_oof_matrix(Xs, y, splits)
        # notebook 05's numbers, reproduced on the same patients, labels and splits
        ref = probe05[mode].set_index(['task_id', 'encoder']).loc[(tid, enc)]
        d_img, d_bg = mean_auc(y, P) - ref.image_auc, mean_auc(y, Pb) - ref.background_auc
        repro = bool(int(ref.n_patients) == len(y) and abs(d_img) < REPRO_TOL and abs(d_bg) < REPRO_TOL)
        # (2) the silhouette probe, and background minus silhouette
        lo, hi = boot_ci(y, Ps)
        obs, p, null = permutation_test(scaled_folds(Xs), y)
        d, d_lo, d_hi = boot_diff(y, Pb, Ps)
        rec['silhouette'].append({**base, 'encoder': enc, 'silhouette_auc': mean_auc(y, Ps), 'silhouette_ci_low': lo,
                                  'silhouette_ci_high': hi, 'silhouette_perm_p': p,
                                  'silhouette_null_mean_auc': float(null.mean()),
                                  'background_auc': mean_auc(y, Pb), 'background_flagged_nb05': bool(ref.background_flagged),
                                  'background_minus_silhouette': d, 'difference_ci_low': d_lo,
                                  'difference_ci_high': d_hi, 'image_auc': mean_auc(y, P),
                                  'body_size_auc_nb05': float(ref.body_size_auc),
                                  'image_auc_minus_nb05': d_img, 'background_auc_minus_nb05': d_bg,
                                  'reproduced_nb05': repro})
        # (1) background (and silhouette, and image) under acquisition control
        for variant, Xv, Pv in (('image', X, P), ('background', Xb, Pb), ('silhouette', Xs, Ps)):
            row = {'task_id': tid, 'encoder': enc, 'variant': variant, 'grouping': grouping,
                   'n_patients': int(keep.sum()), 'overall_auc_same_patients': mean_auc(y[keep], Pv[:, keep])}
            if variant != 'image':        # notebook 05 ran the within-group test for the full image already
                w = within_site_estimate(y[keep], Pv[:, keep], strata[keep])
                if w['n_pairs']:
                    obs_w, p_w, null_w = within_group_permutation_test(scaled_folds(Xv), y, strata, keep)
                else:
                    obs_w, p_w, null_w = np.nan, np.nan, np.array([np.nan])
                row.update({'n_groups_with_both_classes': w['n_sites_with_pairs'], 'n_pairs': w['n_pairs'],
                            'within_auc': w['auc'], 'within_ci_low': w['ci_low'], 'within_ci_high': w['ci_high'],
                            'within_auc_fixed_split': obs_w, 'null_mean_within_auc': float(np.nanmean(null_w)),
                            'within_perm_p': p_w})
            h = heldout(Xv, y, strata, keep)
            row.update({'n_groups': h['n_groups'], 'heldout_auc': h.get('auc', np.nan),
                        'heldout_ci_low': h.get('ci_low', np.nan), 'heldout_ci_high': h.get('ci_high', np.nan),
                        'heldout_note': h.get('note', '')})
            rec['controlled'].append(row)
        # (3) background versus the DICOM header
        P_hb = cv_oof_combined(df, Xb, y, HDR_COLS, splits)
        d3, d3_lo, d3_hi = boot_diff(y, P_hb, P_hdr)
        sb, sh = Pb[:, :, 1].mean(0), P_hdr[:, :, 1].mean(0)
        rec['header'].append({'task_id': tid, 'encoder': enc, 'header_only_auc': mean_auc(y, P_hdr),
                              'background_auc': mean_auc(y, Pb), 'header_plus_background_auc': mean_auc(y, P_hb),
                              'added_by_background': d3, 'added_ci_low': d3_lo, 'added_ci_high': d3_hi,
                              **within_class_spearman(y, sb, sh, classes)})
        for j, c in enumerate(classes):        # (3c) descriptive, per header field
            m = y == j
            for f in HDR_NUMERIC + ['hdr_coverage_mm', 'body_area_cm2']:
                v = df[f].to_numpy(float)[m]
                ok = np.isfinite(v)
                r_ = {'task_id': tid, 'encoder': enc, 'class': c, 'field': f, 'test': 'Spearman rho', 'n': int(ok.sum())}
                if ok.sum() >= 10 and np.unique(v[ok]).size > 1:
                    s = spearmanr(sb[m][ok], v[ok])
                    r_.update({'statistic': float(s.statistic), 'p': float(s.pvalue)})
                rec['fields'].append(r_)
            for f in HDR_CATEGORICAL + ['hdr_manufacturer']:
                g = df[f][m].to_numpy()
                levels = [l for l in pd.Series(g).dropna().unique() if (g == l).sum() >= MIN_COUNT]
                r_ = {'task_id': tid, 'encoder': enc, 'class': c, 'field': f, 'test': 'Kruskal-Wallis H',
                      'n': int(sum((g == l).sum() for l in levels)), 'n_levels': len(levels)}
                if len(levels) >= 2:
                    try:
                        s = kruskal(*[sb[m][g == l] for l in levels])
                        r_.update({'statistic': float(s.statistic), 'p': float(s.pvalue)})
                    except ValueError:      # all scores identical
                        pass
                rec['fields'].append(r_)
        rec['oof'].append(pd.DataFrame({'task_id': tid, 'encoder': enc, 'key': df.key, 'label': y,
                                        'p_image': P[:, :, 1].mean(0), 'p_background': sb,
                                        'p_silhouette': Ps[:, :, 1].mean(0), 'p_header_only': sh}))
    return rec

In [13]:
# 13. Run one mode (rehearsal on shuffled labels, or real), with BH, flags, the reading rule and saving
def reading(bg_flagged, sil_flagged, diff_ci_low):
    """Plan 7f.2, fixed before the run."""
    if not bg_flagged:
        return 'not applicable (background-only not flagged in notebook 05)'
    if sil_flagged:
        return 'may reflect the body outline or positioning'
    if diff_ci_low > 0:
        return 'beyond the body outline'
    return 'unresolved'

def run_mode(mode):
    out = OUT_ROOT / mode
    (out / 'per_task').mkdir(parents=True, exist_ok=True)
    shuffle_seed = 999 if mode == 'rehearsal' else (1000 if TEST.get('real_is_shuffled') else None)
    key = hashlib.md5(json.dumps({'SEED': SEED, 'N_SPLITS': N_SPLITS, 'N_REPEATS': N_REPEATS, 'N_BOOT': N_BOOT,
                                  'N_PERM': N_PERM, 'LR': LR_PARAMS, 'EMB_KEY': EMB_KEY, 'EMB_KEY05': EMB_KEY05,
                                  'TEST': TEST, 'shuffle': shuffle_seed, 'complete': sorted(complete)},
                                 sort_keys=True, default=str).encode()).hexdigest()
    print(f'\n===== {mode.upper()} pass ({"shuffled labels" if shuffle_seed is not None else "REAL labels"}) =====')
    allrec = {'silhouette': [], 'controlled': [], 'header': [], 'fields': []}
    oof = []
    t_all = time.time()
    for i, tid in enumerate(TASKS, 1):
        t0 = time.time()
        stem = re.sub(r'\W+', '_', tid).strip('_')
        saved = out / 'per_task' / f'{stem}.json'
        if saved.exists() and json.loads(saved.read_text()).get('settings_key') == key:
            rec = json.loads(saved.read_text())['rec']
            oof.append(pd.read_csv(out / 'per_task' / f'{stem}_oof.csv.gz'))
            print(f'{i}/{len(TASKS)} {tid:<36} loaded from an earlier run')
        else:
            rec = analyse_task(tid, mode, shuffle_seed)
            o = pd.concat(rec.pop('oof'), ignore_index=True)
            o.to_csv(out / 'per_task' / f'{stem}_oof.csv.gz', index=False)
            oof.append(o)
            write_json_atomic(saved, {'settings_key': key, 'rec': rec})
            s = {r_['encoder']: r_ for r_ in rec['silhouette']}
            print(f"{i}/{len(TASKS)} {tid:<36} silhouette CT-FM {s['ctfm']['silhouette_auc']:.3f} "
                  f"(bg {s['ctfm']['background_auc']:.3f}) | Merlin {s['merlin']['silhouette_auc']:.3f} "
                  f"(bg {s['merlin']['background_auc']:.3f})  ({(time.time() - t0) / 60:.1f} min)")
        for k_ in allrec:
            allrec[k_] += rec[k_]
    print(f'Analysis finished in {(time.time() - t_all) / 60:.1f} min')

    sil = pd.DataFrame(allrec['silhouette'])
    sil['silhouette_perm_q_bh'] = np.nan
    for enc, idx in sil.groupby('encoder').groups.items():
        sil.loc[idx, 'silhouette_perm_q_bh'] = bh(sil.loc[idx, 'silhouette_perm_p'])
    sil['silhouette_flagged'] = [flag(lo, q) for lo, q in zip(sil.silhouette_ci_low, sil.silhouette_perm_q_bh)]
    sil['reading'] = [reading(b, s, d) for b, s, d in zip(sil.background_flagged_nb05, sil.silhouette_flagged,
                                                          sil.difference_ci_low)]
    ctrl = pd.DataFrame(allrec['controlled'])
    ctrl['within_perm_q_bh'] = np.nan
    for (enc, variant), idx in ctrl[ctrl.variant != 'image'].groupby(['encoder', 'variant']).groups.items():
        ctrl.loc[idx, 'within_perm_q_bh'] = bh(ctrl.loc[idx, 'within_perm_p'])
    tables = {'silhouette_probe.csv': sil, 'acquisition_controlled.csv': ctrl,
              'background_vs_header.csv': pd.DataFrame(allrec['header']),
              'background_header_fields.csv': pd.DataFrame(allrec['fields'])}
    for name, t in tables.items():
        t.to_csv(out / name, index=False)
    pd.concat(oof, ignore_index=True).to_csv(out / 'oof_posthoc_predictions.csv.gz', index=False)
    show(sil[['task_id', 'encoder', 'silhouette_auc', 'silhouette_ci_low', 'silhouette_perm_q_bh', 'silhouette_flagged',
              'background_auc', 'background_minus_silhouette', 'difference_ci_low', 'reading']], 20)
    show(ctrl[['task_id', 'encoder', 'variant', 'grouping', 'within_auc', 'within_perm_q_bh', 'heldout_auc',
               'heldout_ci_low', 'heldout_ci_high']], 30)

    outputs = []
    for p in sorted(q for q in out.rglob('*') if q.is_file()):
        if p.name == 'run_manifest.json':
            continue
        n_rows = len(pd.read_csv(p, low_memory=False)) if (p.suffix == '.csv' or p.name.endswith('.csv.gz')) else None
        outputs.append({'file': p.relative_to(out).as_posix(), 'bytes': p.stat().st_size, 'rows': n_rows, 'md5': md5(p)})
    cache_files = sorted((q for q in CACHE.rglob('*') if q.is_file() and q.suffix in ('.npz', '.json', '.csv')),
                         key=lambda q: q.relative_to(CACHE).parts)
    manifest = {
        'settings_key': key, 'notebook': '06_image_posthoc_checks.ipynb', 'mode': mode, 'post_hoc': True,
        'labels': ('real' if shuffle_seed is None else 'shuffled (rehearsal)' if mode == 'rehearsal'
                   else 'shuffled (local code test)'), 'run_started_utc': RUN_STARTED,
        'run_finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        'in_colab': IN_COLAB, 'local_test_settings': TEST,
        'input_runs': {'01': man01['run_started_utc'], '02': man02['run_started_utc'], '03': man03['run_started_utc'],
                       '04': man04['run_started_utc'], '05': man05['real']['run_started_utc']},
        'input_notebook05_cache_md5': cache05_md5,
        'tasks': TASKS, 'same_versions_as_notebook05': same_versions,
        'reproduced_notebook05': bool(sil.reproduced_nb05.all()),
        'patients_total': int(len(PATIENTS)), 'patients_with_silhouette': int(len(complete)),
        'reembedding_check': REEMB,
        'settings': {**same, 'N_BOOT': N_BOOT, 'N_PERM': N_PERM, 'SIL_HU': SIL_HU, 'REPRO_TOL': REPRO_TOL,
                     'PREP_KEY06': PREP_KEY,
                     'EMB_KEY06': EMB_KEY},
        'versions': VERSIONS,
        'cache_files': {'count': len(cache_files),
                        'combined_md5': hashlib.md5(''.join(md5(q) for q in cache_files).encode()).hexdigest()},
        'outputs': outputs,
    }
    (out / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
    return sil, ctrl

In [ ]:
# 14. Run: rehearsal (shuffled labels) -> automatic gate -> real labels
modes = {'both': ['rehearsal', 'real'], 'rehearsal': ['rehearsal'], 'real': ['real']}[RUN_MODE]
results = {}
for mode in modes:
    if mode == 'real' and 'rehearsal' in results:
        sil_r, ctrl_r = results['rehearsal']
        nulls = pd.concat([sil_r.silhouette_null_mean_auc, ctrl_r.null_mean_within_auc.dropna()])
        problems = []
        if len(sil_r) != 2 * len(TASKS):
            problems.append('not every task and encoder finished')
        if not nulls.between(*GATE_NULL_RANGE).all():
            problems.append(f'a null mean AUC is outside {GATE_NULL_RANGE}: {nulls.round(3).tolist()}')
        if (cov['mean'] < GATE_MIN_COVERAGE).any():
            problems.append('too few silhouettes embedded for a task')
        if problems:
            print('\nGATE NOT PASSED; the real-label pass was NOT run:', *problems, sep='\n - ')
            print('Inspect the run log.')
            break
        print('\nRehearsal gate passed: all analyses ran, null means look like chance, enough scans.')
        if not TEST and not sil_r.reproduced_nb05.all():
            print('WARNING: the rehearsal did not reproduce notebook 05 exactly (package versions may have changed). '
                  'The real pass runs, but review the run log before interpreting it.')
    results[mode] = run_mode(mode)


===== REHEARSAL pass (shuffled labels) =====
1/4 renal subtype (KIRC vs KIRP)         silhouette CT-FM 0.507 (bg 0.526) | Merlin 0.532 (bg 0.450)  (6.1 min)
2/4 CPTAC-PDA | stage                    silhouette CT-FM 0.542 (bg 0.400) | Merlin 0.359 (bg 0.446)  (2.7 min)
3/4 lung histology (LUAD vs LUSC)        silhouette CT-FM 0.472 (bg 0.414) | Merlin 0.490 (bg 0.549)  (3.5 min)
4/4 LIDC-IDRI | primary vs metastatic    silhouette CT-FM 0.554 (bg 0.600) | Merlin 0.452 (bg 0.539)  (3.0 min)
Analysis finished in 15.3 min


,task_id,encoder,silhouette_auc,silhouette_ci_low,silhouette_perm_q_bh,silhouette_flagged,background_auc,background_minus_silhouette,difference_ci_low,reading
0,renal subtype (KIRC vs KIRP),ctfm,0.506669,0.397489,0.907,False,0.525718,0.019049,-0.100623,not applicable (background-only not flagged in notebook 05)
1,renal subtype (KIRC vs KIRP),merlin,0.532461,0.405773,0.884,False,0.450295,-0.082167,-0.272280,not applicable (background-only not flagged in notebook 05)
2,CPTAC-PDA | stage,ctfm,0.542452,0.417337,0.907,False,0.399595,-0.142857,-0.285933,not applicable (background-only not flagged in notebook 05)
3,CPTAC-PDA | stage,merlin,0.358663,0.242705,0.962,False,0.445795,0.087133,-0.045406,not applicable (background-only not flagged in notebook 05)
4,lung histology (LUAD vs LUSC),ctfm,0.472287,0.374376,0.907,False,0.413796,-0.058491,-0.154739,not applicable (background-only not flagged in notebook 05)
5,lung histology (LUAD vs LUSC),merlin,0.490359,0.389998,0.884,False,0.549415,0.059056,-0.056156,not applicable (background-only not flagged in notebook 05)
6,LIDC-IDRI | primary vs metastatic,ctfm,0.554127,0.451513,0.907,False,0.600274,0.046147,-0.067136,not applicable (background-only not flagged in notebook 05)
7,LIDC-IDRI | primary vs metastatic,merlin,0.451892,0.346635,0.884,False,0.539261,0.087369,-0.063781,not applicable (background-only not flagged in notebook 05)


,task_id,encoder,variant,grouping,within_auc,within_perm_q_bh,heldout_auc,heldout_ci_low,heldout_ci_high
0,renal subtype (KIRC vs KIRP),ctfm,image,hospital,NaN,NaN,0.467944,0.336428,0.604817
1,renal subtype (KIRC vs KIRP),ctfm,background,hospital,0.484822,0.970667,0.463154,0.342384,0.583828
2,renal subtype (KIRC vs KIRP),ctfm,silhouette,hospital,0.442842,0.908000,0.489315,0.351937,0.623047
3,renal subtype (KIRC vs KIRP),merlin,image,hospital,NaN,NaN,0.532424,0.395029,0.672692
4,renal subtype (KIRC vs KIRP),merlin,background,hospital,0.466523,0.633333,0.279293,0.157515,0.418128
5,renal subtype (KIRC vs KIRP),merlin,silhouette,hospital,0.512594,0.922667,0.502579,0.371257,0.640270
6,CPTAC-PDA | stage,ctfm,image,description language,NaN,NaN,0.595745,0.449174,0.724310
7,CPTAC-PDA | stage,ctfm,background,description language,0.419101,0.970667,0.485309,0.341328,0.642310
8,CPTAC-PDA | stage,ctfm,silhouette,description language,0.551685,0.908000,0.612969,0.465079,0.757291
9,CPTAC-PDA | stage,merlin,image,description language,NaN,NaN,0.452888,0.302176,0.602094



Rehearsal gate passed: all analyses ran, null means look like chance, enough scans.

===== REAL pass (REAL labels) =====
1/4 renal subtype (KIRC vs KIRP)         silhouette CT-FM 0.654 (bg 0.617) | Merlin 0.609 (bg 0.624)  (5.6 min)
2/4 CPTAC-PDA | stage                    silhouette CT-FM 0.572 (bg 0.563) | Merlin 0.313 (bg 0.509)  (2.8 min)
3/4 lung histology (LUAD vs LUSC)        silhouette CT-FM 0.588 (bg 0.696) | Merlin 0.571 (bg 0.821)  (3.4 min)
4/4 LIDC-IDRI | primary vs metastatic    silhouette CT-FM 0.672 (bg 0.719) | Merlin 0.689 (bg 0.726)  (2.9 min)
Analysis finished in 14.7 min


,task_id,encoder,silhouette_auc,silhouette_ci_low,silhouette_perm_q_bh,silhouette_flagged,background_auc,background_minus_silhouette,difference_ci_low,reading
0,renal subtype (KIRC vs KIRP),ctfm,0.654348,0.549629,0.093333,False,0.617097,-0.037251,-0.138150,not applicable (background-only not flagged in notebook 05)
1,renal subtype (KIRC vs KIRP),merlin,0.609359,0.484340,0.100000,False,0.623618,0.014259,-0.144041,not applicable (background-only not flagged in notebook 05)
2,CPTAC-PDA | stage,ctfm,0.572239,0.442004,0.093333,False,0.562918,-0.009321,-0.151482,not applicable (background-only not flagged in notebook 05)
3,CPTAC-PDA | stage,merlin,0.313475,0.212309,0.935000,False,0.508612,0.195137,0.042079,not applicable (background-only not flagged in notebook 05)
4,lung histology (LUAD vs LUSC),ctfm,0.587818,0.494525,0.235000,False,0.695845,0.108027,-0.007713,not applicable (background-only not flagged in notebook 05)
5,lung histology (LUAD vs LUSC),merlin,0.571037,0.463930,0.153333,False,0.821380,0.250343,0.136761,beyond the body outline
6,LIDC-IDRI | primary vs metastatic,ctfm,0.672047,0.572230,0.028000,False,0.718833,0.046785,-0.049214,unresolved
7,LIDC-IDRI | primary vs metastatic,merlin,0.689193,0.581281,0.008000,False,0.726037,0.036845,-0.080779,unresolved


,task_id,encoder,variant,grouping,within_auc,within_perm_q_bh,heldout_auc,heldout_ci_low,heldout_ci_high
0,renal subtype (KIRC vs KIRP),ctfm,image,hospital,NaN,NaN,0.319455,0.203419,0.453582
1,renal subtype (KIRC vs KIRP),ctfm,background,hospital,0.595902,0.626000,0.335667,0.220559,0.462217
2,renal subtype (KIRC vs KIRP),ctfm,silhouette,hospital,0.646721,0.172000,0.469971,0.327400,0.617108
3,renal subtype (KIRC vs KIRP),merlin,image,hospital,NaN,NaN,0.301769,0.205594,0.412944
4,renal subtype (KIRC vs KIRP),merlin,background,hospital,0.571311,0.276000,0.488578,0.377490,0.608155
5,renal subtype (KIRC vs KIRP),merlin,silhouette,hospital,0.631967,0.092000,0.412307,0.300067,0.524208
6,CPTAC-PDA | stage,ctfm,image,description language,NaN,NaN,0.443769,0.290774,0.594801
7,CPTAC-PDA | stage,ctfm,background,description language,0.497312,0.196000,0.319149,0.186432,0.455215
8,CPTAC-PDA | stage,ctfm,silhouette,description language,0.528495,0.213333,0.449848,0.296436,0.607014
9,CPTAC-PDA | stage,merlin,image,description language,NaN,NaN,0.303951,0.182222,0.437500


In [ ]:
# 15. Plain-English summary, read back from the saved files
for mode in results:
    out = OUT_ROOT / mode
    sl = pd.read_csv(out / 'silhouette_probe.csv')
    ct = pd.read_csv(out / 'acquisition_controlled.csv')
    lab_kind = json.loads((out / 'run_manifest.json').read_text())['labels']
    print(f"\n{mode.upper()} ({lab_kind}{': every number should look like chance' if lab_kind.startswith('shuffled') else ''})")
    for enc, g in sl.groupby('encoder'):
        print(f"- {enc}: silhouette flagged in {int(g.silhouette_flagged.sum())} of {len(g)} tasks; "
              f"silhouette null means {g.silhouette_null_mean_auc.min():.3f}-{g.silhouette_null_mean_auc.max():.3f}; "
              f"notebook 05 reproduced: {g.reproduced_nb05.tolist()} (largest AUC difference "
              f"{g[['image_auc_minus_nb05', 'background_auc_minus_nb05']].abs().max().max():.1e})")
    for r in sl[sl.background_flagged_nb05].itertuples():
        print(f"- {r.task_id} / {r.encoder}: background {r.background_auc:.3f}, silhouette {r.silhouette_auc:.3f} "
              f"-> {r.reading}")
    bg = ct[(ct.variant == 'background') & ct.within_perm_q_bh.notna()]
    print(f"- background inside one group: q < 0.05 for {int((bg.within_perm_q_bh < 0.05).sum())} of {len(bg)} "
          f"task-encoder pairs")
print('\nNext: review the saved results and run log before writing up findings.')


REHEARSAL (shuffled (rehearsal): every number should look like chance)
- ctfm: silhouette flagged in 0 of 4 tasks; silhouette null means 0.488-0.495; notebook 05 reproduced: [True, True, True, True] (largest AUC difference 5.6e-17)
- merlin: silhouette flagged in 0 of 4 tasks; silhouette null means 0.489-0.494; notebook 05 reproduced: [True, True, True, True] (largest AUC difference 5.6e-17)
- background inside one group: q < 0.05 for 0 of 8 task-encoder pairs

REAL (real)
- ctfm: silhouette flagged in 0 of 4 tasks; silhouette null means 0.485-0.492; notebook 05 reproduced: [True, True, True, True] (largest AUC difference 0.0e+00)
- merlin: silhouette flagged in 0 of 4 tasks; silhouette null means 0.486-0.493; notebook 05 reproduced: [True, True, True, True] (largest AUC difference 5.6e-17)
- lung histology (LUAD vs LUSC) / merlin: background 0.821, silhouette 0.571 -> beyond the body outline
- LIDC-IDRI | primary vs metastatic / ctfm: background 0.719, silhouette 0.672 -> unresolved
